Open in colab: https://colab.research.google.com/github/Thanaritt-K/Thai-Lang-Emb-Bias-experiment/blob/main/Lang_Emb_Bias_experiment.ipynb

First, we need to install some dependencies used in this experiment

In [2]:
import numpy as np
from scipy.spatial.distance import cosine
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

The model is readily available for testing at https://huggingface.co/typhoon-ai/typhoon-s-thaillm-8b-instruct-research-preview

In [3]:
model_id = "typhoon-ai/typhoon-s-thaillm-8b-instruct-research-preview"

t = AutoTokenizer.from_pretrained(model_id)
t.pad_token = t.eos_token
m = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)
m.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151643)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
      

Weighted-Mean-Pooling as presented in https://stackoverflow.com/questions/76926025/sentence-embeddings-from-llama-2-huggingface-opensource

In [4]:
texts = [
    "this is a test",
    "this is another test case with a different length",
]
t_input = t(texts, padding=True, return_tensors="pt")


with torch.no_grad():
    last_hidden_state = m(**t_input, output_hidden_states=True).hidden_states[-1]


weights_for_non_padding = t_input.attention_mask * torch.arange(start=1, end=last_hidden_state.shape[1] + 1).unsqueeze(0)

sum_embeddings = torch.sum(last_hidden_state * weights_for_non_padding.unsqueeze(-1), dim=1)
num_of_none_padding_tokens = torch.sum(weights_for_non_padding, dim=-1).unsqueeze(-1)
sentence_embeddings = sum_embeddings / num_of_none_padding_tokens

print(t_input.input_ids)
print(weights_for_non_padding)
print(num_of_none_padding_tokens)
print(sentence_embeddings.shape)

tensor([[151645, 151645, 151645, 151645, 151645,    574,    374,    264,   1273],
        [   574,    374,   2441,   1273,   1142,    448,    264,   2155,   3084]])
tensor([[0, 0, 0, 0, 0, 6, 7, 8, 9],
        [1, 2, 3, 4, 5, 6, 7, 8, 9]])
tensor([[30],
        [45]])
torch.Size([2, 4096])


Defining Thai terms equivalent to previous studies, as well as, semantically bleached sentence templates.

In [5]:
# Define WEAT word sets (Example: Gender bias test)
#A = ["ชาย", "ผู้ชาย", "พี่ชาย", "น้องชาย", "บุรุษ"]  # Male terms
#B = ["หญิง", "สาว", "ผู้หญิง", "น้องสาว", "พี่สาว"]  # Female terms

#X = ["งาน", "ธุรกิจ", "อาชีพ", "บริษัท", "เงินเดือน", "บริหาร", "มืออาชีพ"]  # Career-related
#Y = ["บ้าน", "ลูก", "หลาน", "ครอบครัว", "พ่อแม่", "ญาติ","แต่งงาน"]  # Family-related

#Frame = ["เขาคือ", "คนคนนั้นเป็น","คนนี้คือ"] # Semantically bleached sentence templates

# Simplified version of experiment
A = ["ชาย", "ผู้ชาย", "บุรุษ", "พี่ชาย"]  # Male terms
B = ["หญิง", "สาว", "ผู้หญิง", "พี่สาว"]  # Female terms

X = ["งาน", "ธุรกิจ", "อาชีพ","เงินเดือน","มืออาชีพ", "บริษัท"]  # Career-related
Y = ["บ้าน", "ลูก", "ครอบครัว", "พ่อแม่", "ญาติ", "แต่งงาน"]  # Family-related

Frame = ["มี", "นั่นเป็น"] # Semantically bleached sentence templates

In [6]:
# Generating SEAT sentence sets
Sen_A = [f + a for a in A for f in Frame]
Sen_B = [f + b for b in B for f in Frame]
Sen_X = [f + x for x in X for f in Frame]
Sen_Y = [f + y for y in Y for f in Frame]
print(Sen_A)

['มีชาย', 'นั่นเป็นชาย', 'มีผู้ชาย', 'นั่นเป็นผู้ชาย', 'มีบุรุษ', 'นั่นเป็นบุรุษ', 'มีพี่ชาย', 'นั่นเป็นพี่ชาย']


Defining a function to get sentence embeddings

In [7]:
# Function to get embeddings
def get_embedding(texts):
    t_input = t(texts, padding=True, return_tensors="pt")


    with torch.no_grad():
      last_hidden_state = m(**t_input, output_hidden_states=True).hidden_states[-1]


    weights_for_non_padding = t_input.attention_mask * torch.arange(start=1, end=last_hidden_state.shape[1] + 1).unsqueeze(0)

    sum_embeddings = torch.sum(last_hidden_state * weights_for_non_padding.unsqueeze(-1), dim=1)
    num_of_none_padding_tokens = torch.sum(weights_for_non_padding, dim=-1).unsqueeze(-1)
    sentence_embeddings = sum_embeddings / num_of_none_padding_tokens
    return sentence_embeddings

Checking sentence embedding of an example test sentence.

In [8]:
#Checking the embedding
embedding_1=get_embedding(Sen_A[0])
print(f"Vector Shape: {embedding_1.shape}")
print(f"First 5 Values: {embedding_1[:5]}")

Vector Shape: torch.Size([1, 4096])
First 5 Values: tensor([[ 0.3066, -0.4473,  0.0767,  ...,  0.5547,  0.4941, -0.1484]],
       dtype=torch.bfloat16)


Retrieve sentence embeddings for SEAT sentence sets

In [10]:
from tqdm.auto import tqdm
embedding = {}
for sen in tqdm(Sen_A):
  embedding[sen] = get_embedding(sen)

print(embedding[Sen_A[0]])

  0%|          | 0/8 [00:00<?, ?it/s]

tensor([[ 0.3066, -0.4473,  0.0767,  ...,  0.5547,  0.4941, -0.1484]],
       dtype=torch.bfloat16)


In [11]:
for sen in tqdm(Sen_B):
  embedding[sen] = get_embedding(sen)

  0%|          | 0/8 [00:00<?, ?it/s]

In [12]:
for sen in tqdm(Sen_X):
  embedding[sen] = get_embedding(sen)

  0%|          | 0/12 [00:00<?, ?it/s]

In [13]:
for sen in tqdm(Sen_Y):
  embedding[sen] = get_embedding(sen)

  0%|          | 0/12 [00:00<?, ?it/s]

Defining function to calculate cosine similarity

In [14]:
import scipy

# Compute mean cosine similarity
def mean_cosine_similarity(w, A, B, word_embedding, all_sen):

    if w in all_sen:
        return all_sen[w]

    A_sim = np.mean([scipy.spatial.distance.cosine(word_embedding[w][0].float().numpy(), word_embedding[a][0].float().numpy()) for a in A])
    #print (A_sim)
    B_sim = np.mean([scipy.spatial.distance.cosine(word_embedding[w][0].float().numpy(), word_embedding[b][0].float().numpy()) for b in B])
    #print (B_sim)

    all_sen[w] = A_sim - B_sim
    return all_sen[w]

Testing cosine distance function of two examples from 2 attributive sentences.

In [15]:
all_sen = {}
a=mean_cosine_similarity(Sen_X[0], Sen_A, Sen_B, embedding, all_sen)
print(a)

0.007648267


In [16]:
a=mean_cosine_similarity(Sen_Y[0], Sen_A, Sen_B, embedding, all_sen)
print(a)

0.002698116


Defining function to calculate SEAT effect size

In [17]:
# Compute SEAT effect size
def seat_effect_size(X, Y, A, B, word_embedding, all_sen):
    s_x = np.array([mean_cosine_similarity(x, A, B, word_embedding, all_sen) for x in X])
    #print (x)
    s_y = np.array([mean_cosine_similarity(y, A, B, word_embedding, all_sen) for y in Y])
    #print (s_y)
    return (np.mean(s_x) - np.mean(s_y)) / np.std(np.concatenate((s_x, s_y)))

Testing SEAT effect size function of two concept sets

In [18]:
# Run SEAT test
effect_size = seat_effect_size(Sen_X,Sen_Y, Sen_A, Sen_B, embedding, all_sen)
print(f"SEAT Effect Size: {effect_size}")

SEAT Effect Size: -0.33758145570755005


Defining function to calculate P Value using a permutation test

In [19]:
import itertools

def s_group(X, Y, A, B, word_embedding, all_sen):

    total = 0
    for x in X:
        total += mean_cosine_similarity(x, A, B, word_embedding, all_sen)
    for y in Y:
        total -= mean_cosine_similarity(y, A, B, word_embedding, all_sen)
    return total

def p_value_exhust(X, Y, A, B, word_embedding):

    all_sen = {}
    s_orig = s_group(X, Y, A, B, word_embedding, all_sen)

    union = set(X+Y)
    subset_size = int(len(union)/2)

    larger = 0
    total = 0

    combiset = set(itertools.combinations(union, subset_size))
    for subset in tqdm(combiset,total=len(combiset)):
        total += 1
        Xi = list(set(subset))
        Yi = list(union - set(subset))
        if s_group(Xi, Yi, A, B, word_embedding, all_sen) > s_orig:
            larger += 1

    return larger/float(total)

In [20]:
# Run P-value test
p_value = p_value_exhust(Sen_X, Sen_Y, Sen_A, Sen_B, embedding)
print(f"P Value: {p_value}")

  0%|          | 0/2704156 [00:00<?, ?it/s]

P Value: 0.7803762061064524
